In [2]:
import scipy.io as sio
import numpy as np

# Load subject 1, exercise 1
data = sio.loadmat('data/DB5_s1/S1_E1_A1.mat')
print(data.keys())  # see what's inside

emg = data['emg']        # shape: (samples, 16 channels)
labels = data['restimulus']  # gesture label per sample
print(emg.shape, labels.shape)

dict_keys(['__header__', '__version__', '__globals__', 'emg', 'acc', 'stimulus', 'glove', 'subject', 'exercise', 'repetition', 'restimulus', 'rerepetition', 'age', 'circumference', 'frequency', 'gender', 'height', 'weight', 'laterality', 'sensor'])
(130267, 16) (130267, 1)


In [5]:
import scipy.io as sio
import numpy as np
import pickle
from scipy.signal import butter, filtfilt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report

def bandpass_filter(signal, lowcut=20, highcut=90, fs=200, order=4):
    nyq = fs / 2
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, np.array(signal), axis=0)

def extract_features(window):
    window = np.array(window)
    mav = np.mean(np.abs(window), axis=0)
    rms = np.sqrt(np.mean(window**2, axis=0))
    zc = np.sum(np.diff(np.sign(window), axis=0) != 0, axis=0)
    wl = np.sum(np.abs(np.diff(window, axis=0)), axis=0)
    return np.concatenate([mav, rms, zc, wl])

def build_dataset(emg, labels, window_size=200, step=50):
    X, y = [], []
    filtered = bandpass_filter(emg)
    for i in range(0, len(filtered) - window_size, step):
        window = filtered[i:i+window_size]
        label = labels[i + window_size//2][0]
        if label == 0 or label > 6:
            continue
        X.append(extract_features(window))
        y.append(label)
    return np.array(X), np.array(y)

# Load all 10 subjects, all 3 exercises
all_X, all_y = [], []
for s in range(1, 11):
    for ex in ['E1', 'E2', 'E3']:
        try:
            path = f'data/DB5_s{s}/S{s}_{ex}_A1.mat'
            mat = sio.loadmat(path)
            emg = mat['emg']
            labels = mat['restimulus']
            X, y = build_dataset(emg, labels)
            mask = y <= 6
            all_X.append(X[mask])
            all_y.append(y[mask])
            print(f"S{s} {ex}: {X[mask].shape[0]} samples")
        except Exception as e:
            print(f"S{s} {ex} skipped: {e}")

X_all = np.concatenate(all_X)
y_all = np.concatenate(all_y)
print(f"\nTotal: {X_all.shape[0]} samples across {len(np.unique(y_all))} gestures")

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# Hyperparameter search
param_dist = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [None, 20, 30, 40],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 'log2', 0.4],
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    verbose=2,
    random_state=42,
    n_jobs=-1
)
search.fit(X_train, y_train)

print(f"\nBest params: {search.best_params_}")
print(f"CV accuracy: {search.best_score_:.2%}")

best_model = search.best_estimator_
print(f"Test accuracy: {accuracy_score(y_test, best_model.predict(X_test)):.2%}")
print(classification_report(y_test, best_model.predict(X_test)))

with open('model/gesture_classifier.pkl', 'wb') as f:
    pickle.dump(best_model, f)

pipeline = {
    'window_size': 200, 'step': 50,
    'gestures': list(np.unique(y_all)),
    'gesture_names': {1: 'index flex', 2: 'middle flex', 3: 'ring flex',
                      4: 'pinky flex', 5: 'thumb flex', 6: 'fist'}
}
with open('model/pipeline_config.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print("Saved.")

S1 E1: 520 samples
S1 E2: 536 samples
S1 E3: 540 samples
S2 E1: 663 samples
S2 E2: 482 samples
S2 E3: 485 samples
S3 E1: 510 samples
S3 E2: 484 samples
S3 E3: 500 samples
S4 E1: 575 samples
S4 E2: 514 samples
S4 E3: 501 samples
S5 E1: 522 samples
S5 E2: 500 samples
S5 E3: 589 samples
S6 E1: 674 samples
S6 E2: 592 samples
S6 E3: 562 samples
S7 E1: 556 samples
S7 E2: 539 samples
S7 E3: 558 samples
S8 E1: 539 samples
S8 E2: 474 samples
S8 E3: 573 samples
S9 E1: 545 samples
S9 E2: 478 samples
S9 E3: 531 samples
S10 E1: 673 samples
S10 E2: 542 samples
S10 E3: 512 samples

Total: 16269 samples across 6 gestures
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END max_depth=40, max_features=0.4, min_samples_leaf=3, min_samples_split=4, n_estimators=200; total time=  36.5s
[CV] END max_depth=40, max_features=0.4, min_samples_leaf=3, min_samples_split=4, n_estimators=200; total time=  36.6s
[CV] END max_depth=40, max_features=0.4, min_samples_leaf=3, min_samples_split=4, n_esti